Data Transaksi

In [ ]:
import pandas as pd
from itertools import combinations

# Data transaksi (diambil dari praktikum sebelumnya)
transactions = [
    ["Laptop", "Mouse", "Keyboard", "Headset"],        # T001
    ["Handphone", "Charger", "Casing HP"],             # T002
    ["Laptop", "Mouse", "Flashdisk"],                  # T003
    ["Handphone", "Headset", "Power Bank", "Charger"], # T004
    ["Tablet", "Keyboard", "Mouse"],                   # T005
    ["Laptop", "Keyboard", "Mouse", "Headset"],        # T006
    ["Handphone", "Casing HP", "Flashdisk", "Charger"],# T007
    ["Laptop", "Mouse", "Headset", "Power Bank"],      # T008
    ["Handphone", "Charger", "Power Bank"],            # T009
    ["Tablet", "Mouse", "Headset", "Flashdisk"]        # T010
]

total_transaksi = len(transactions)

Hitung Support 1-Itemset

In [2]:
from collections import Counter

# Hitung frekuensi item
item_count = Counter()

for trx in transactions:
    item_count.update(trx)

# Buat DataFrame
df_1item = pd.DataFrame([
    {"Item": item, "Frekuensi": freq, "Support": freq/total_transaksi}
    for item, freq in item_count.items()
])

print("=== 1-Itemset ===")
print(df_1item.sort_values(by="Support", ascending=False))

=== 1-Itemset ===
         Item  Frekuensi  Support
1       Mouse          6      0.6
3     Headset          5      0.5
0      Laptop          4      0.4
4   Handphone          4      0.4
5     Charger          4      0.4
2    Keyboard          3      0.3
7   Flashdisk          3      0.3
8  Power Bank          3      0.3
6   Casing HP          2      0.2
9      Tablet          2      0.2


Hitung Support 2-Itemset

In [3]:
pair_count = Counter()

for trx in transactions:
    pairs = combinations(trx, 2)
    pair_count.update(pairs)

df_2item = pd.DataFrame([
    {"Itemset": pair, "Frekuensi": freq, "Support": freq/total_transaksi}
    for pair, freq in pair_count.items()
])

print("\n=== 2-Itemset ===")
print(df_2item.sort_values(by="Support", ascending=False))


=== 2-Itemset ===
                    Itemset  Frekuensi  Support
0           (Laptop, Mouse)          4      0.4
4          (Mouse, Headset)          4      0.4
6      (Handphone, Charger)          4      0.4
2         (Laptop, Headset)          3      0.3
1        (Laptop, Keyboard)          2      0.2
12  (Handphone, Power Bank)          2      0.2
5       (Keyboard, Headset)          2      0.2
7    (Handphone, Casing HP)          2      0.2
10       (Mouse, Flashdisk)          2      0.2
17          (Tablet, Mouse)          2      0.2
18        (Keyboard, Mouse)          2      0.2
13    (Headset, Power Bank)          2      0.2
3         (Mouse, Keyboard)          1      0.1
11     (Handphone, Headset)          1      0.1
9       (Laptop, Flashdisk)          1      0.1
8      (Charger, Casing HP)          1      0.1
14       (Headset, Charger)          1      0.1
16       (Tablet, Keyboard)          1      0.1
15    (Power Bank, Charger)          1      0.1
19   (Handphone, Flas

Filter dengan Minimum Support

In [4]:
min_support = 0.3

df_2item_filtered = df_2item[df_2item["Support"] >= min_support]

print("\n=== 2-Itemset (Filtered) ===")
print(df_2item_filtered)


=== 2-Itemset (Filtered) ===
                Itemset  Frekuensi  Support
0       (Laptop, Mouse)          4      0.4
2     (Laptop, Headset)          3      0.3
4      (Mouse, Headset)          4      0.4
6  (Handphone, Charger)          4      0.4


Hitung Confidence & Lift

In [5]:
rules = []

for _, row in df_2item_filtered.iterrows():
    itemA, itemB = row["Itemset"]
    support_ab = row["Support"]

    support_a = item_count[itemA] / total_transaksi
    support_b = item_count[itemB] / total_transaksi

    # Rule A => B
    conf_ab = support_ab / support_a
    lift_ab = conf_ab / support_b

    # Rule B => A
    conf_ba = support_ab / support_b
    lift_ba = conf_ba / support_a

    rules.append({
        "Rule": f"{itemA} => {itemB}",
        "Support": support_ab,
        "Confidence": conf_ab,
        "Lift": lift_ab
    })

    rules.append({
        "Rule": f"{itemB} => {itemA}",
        "Support": support_ab,
        "Confidence": conf_ba,
        "Lift": lift_ba
    })

df_rules = pd.DataFrame(rules)

print("\n=== Association Rules ===")
print(df_rules)


=== Association Rules ===
                   Rule  Support  Confidence      Lift
0       Laptop => Mouse      0.4    1.000000  1.666667
1       Mouse => Laptop      0.4    0.666667  1.666667
2     Laptop => Headset      0.3    0.750000  1.500000
3     Headset => Laptop      0.3    0.600000  1.500000
4      Mouse => Headset      0.4    0.666667  1.333333
5      Headset => Mouse      0.4    0.800000  1.333333
6  Handphone => Charger      0.4    1.000000  2.500000
7  Charger => Handphone      0.4    1.000000  2.500000


Filter dengan Minimum Confidence

In [6]:
min_confidence = 0.5

df_rules_valid = df_rules[
    (df_rules["Support"] >= min_support) &
    (df_rules["Confidence"] >= min_confidence)
]

print("\n=== VALID RULES ===")
print(df_rules_valid.sort_values(by="Lift", ascending=False))


=== VALID RULES ===
                   Rule  Support  Confidence      Lift
6  Handphone => Charger      0.4    1.000000  2.500000
7  Charger => Handphone      0.4    1.000000  2.500000
0       Laptop => Mouse      0.4    1.000000  1.666667
1       Mouse => Laptop      0.4    0.666667  1.666667
3     Headset => Laptop      0.3    0.600000  1.500000
2     Laptop => Headset      0.3    0.750000  1.500000
5      Headset => Mouse      0.4    0.800000  1.333333
4      Mouse => Headset      0.4    0.666667  1.333333
